Downloading all data from `yahoo finance` and `FRED` =)

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import yfinance as yf
import requests
from pandas_datareader import data as pdr


START = "1995-01-01"
END   = "2025-12-01"
MARKET = "SPY"
CONTEXT_LENGTH = 256

FRED_SERIES = {
    "DFF": "fed_funds",
    "DGS10": "rate_10y",
    "DGS2": "rate_2y",
    "T10Y2Y": "yc_slope_10y2y",
    "CPIAUCSL": "cpi",
    "UNRATE": "unemp",
    "INDPRO": "indpro",
    "VIXCLS": "vix",
    "DBAA": "dbaa",
    "DAAA": "daaa",
}

SPLITS = {
    "gfc_2008_2010": ("2008-01-01", "2010-12-31"),
    "covid_2020_2021": ("2020-01-01", "2021-12-31"),
    "inflation_2022_2023": ("2022-01-01", "2023-12-31"),
}

OUT_ROOT = "dataset_finance_fred_full"


def ensure_dirs():
    os.makedirs(os.path.join(OUT_ROOT, "eval_csv"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, "eval_csv_with_context"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, "eval_jsonl"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, "eval_jsonl_with_context"), exist_ok=True)
    os.makedirs(os.path.join(OUT_ROOT, "metadata"), exist_ok=True)


def get_sp500_tickers() -> list[str]:
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/91.0.4472.124 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    tickers = pd.read_html(response.text)[0]["Symbol"].astype(str).tolist()
    return [t.replace(".", "-") for t in tickers]


def download_close(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    px = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    price_field = "Close"

    if isinstance(px.columns, pd.MultiIndex):
        close = pd.concat(
            {t: px[t][price_field] for t in tickers if t in px.columns.get_level_values(0)},
            axis=1,
        )
    else:
        close = px[price_field].to_frame(tickers[0])

    return close.sort_index()


def log_returns(close_df: pd.DataFrame) -> pd.DataFrame:
    return np.log(close_df).diff()


def download_fred(series_map: dict, start: str, end: str) -> pd.DataFrame:
    macro = pdr.DataReader(list(series_map.keys()), "fred", start, end)
    return macro.rename(columns=series_map).sort_index()


def transform_macro(macro: pd.DataFrame) -> pd.DataFrame:
    macro = macro.copy()

    if "cpi" in macro.columns:
        macro["cpi"] = macro["cpi"].pct_change(12)

    if "indpro" in macro.columns:
        macro["indpro"] = macro["indpro"].pct_change(12)

    if "dbaa" in macro.columns and "daaa" in macro.columns:
        macro["baa_aaa_spread"] = macro["dbaa"] - macro["daaa"]

    macro = macro.drop(columns=[c for c in ["dbaa", "daaa"] if c in macro.columns], errors="ignore")
    return macro


def build_full_panel():
    tickers = get_sp500_tickers()
    all_tickers = sorted(set(tickers + [MARKET]))

    close = download_close(all_tickers, START, END)
    rets = log_returns(close)

    market_ret = rets[MARKET].rename("market_ret_1d")
    firm_rets = rets.drop(columns=[MARKET], errors="ignore")

    macro = download_fred(FRED_SERIES, START, END)
    macro = transform_macro(macro)

    idx = rets.index
    macro = macro.reindex(idx).ffill().shift(1)

    panel = firm_rets.copy()
    panel["market_ret_1d"] = market_ret
    for c in macro.columns:
        panel[c] = macro[c]

    panel = panel.sort_index()

    return tickers, firm_rets, market_ret, macro, panel


def get_split_with_context_index(index: pd.DatetimeIndex, split_start: str, split_end: str, context_length: int):
    split_start = pd.to_datetime(split_start)
    split_end = pd.to_datetime(split_end)

    idx = pd.Index(index)
    eval_mask = (idx >= split_start) & (idx <= split_end)
    eval_positions = np.where(eval_mask)[0]

    if len(eval_positions) == 0:
        return None, None, None

    first_eval_pos = int(eval_positions[0])
    last_eval_pos = int(eval_positions[-1])

    start_pos_with_context = max(0, first_eval_pos - context_length)

    used_index = idx[start_pos_with_context:last_eval_pos + 1]
    eval_start_in_used = first_eval_pos - start_pos_with_context

    return used_index, eval_start_in_used, len(eval_positions)


def write_eval_csvs(panel: pd.DataFrame):
    for split_name, (a, b) in SPLITS.items():
        used_index, eval_start_in_used, eval_len = get_split_with_context_index(
            panel.index, a, b, CONTEXT_LENGTH
        )
        if used_index is None:
            print(f"[{split_name}] skipped: no rows in split")
            continue

        # plain eval csv
        plain_df = panel.loc[a:b].copy()
        plain_df = plain_df.reset_index().rename(columns={"index": "date"})
        plain_df["is_eval_target"] = 1
        plain_df.to_csv(
            os.path.join(OUT_ROOT, "eval_csv", f"{split_name}.csv"),
            index=False,
        )

        # with context csv
        ctx_df = panel.loc[used_index].copy()
        ctx_df = ctx_df.reset_index().rename(columns={"index": "date"})
        ctx_df["is_eval_target"] = 0
        ctx_df.loc[eval_start_in_used:, "is_eval_target"] = 1
        ctx_df.to_csv(
            os.path.join(OUT_ROOT, "eval_csv_with_context", f"{split_name}_with_context.csv"),
            index=False,
        )

        print(
            f"[{split_name}] csv rows: plain={len(plain_df)}, "
            f"with_context={len(ctx_df)}, context_rows={eval_start_in_used}, eval_rows={eval_len}"
        )


def write_eval_jsonl_for_split(
    split_name: str,
    split_range: tuple[str, str],
    firm_rets: pd.DataFrame,
    market_ret: pd.Series,
    macro: pd.DataFrame,
    out_path_plain: str,
    out_path_ctx: str,
    min_len: int = 32,
):
    split_start, split_end = split_range

    used_index, eval_start_in_used, eval_len = get_split_with_context_index(
        firm_rets.index, split_start, split_end, CONTEXT_LENGTH
    )
    if used_index is None:
        print(f"[{split_name}] skipped jsonl: no rows in split")
        return

    n_plain = 0
    n_ctx = 0

    with open(out_path_plain, "w", encoding="utf-8") as f_plain, \
         open(out_path_ctx, "w", encoding="utf-8") as f_ctx:

        for t in firm_rets.columns:
            df = pd.DataFrame(index=firm_rets.index)
            df["sequence"] = firm_rets[t]
            df["market_ret_1d"] = market_ret
            for c in macro.columns:
                df[c] = macro[c]

            # plain eval segment
            df_plain = df.loc[split_start:split_end].dropna().copy()

            if len(df_plain) >= min_len:
                obj_plain = {
                    "ticker": t,
                    "segment_id": 0,
                    "dates": [d.strftime("%Y-%m-%d") for d in df_plain.index],
                    "sequence": df_plain["sequence"].astype(float).tolist(),
                    "main_features": df_plain[["sequence", "market_ret_1d"]].astype(float).values.tolist(),
                    "macro_features": df_plain[list(macro.columns)].astype(float).values.tolist(),
                    "eval_start_date": df_plain.index[0].strftime("%Y-%m-%d"),
                    "eval_length": len(df_plain),
                    "context_length": 0,
                }
                f_plain.write(json.dumps(obj_plain) + "\n")
                n_plain += 1

            # eval with context
            df_ctx = df.loc[used_index].copy().dropna().copy()

            # after dropna the split position may shift, so recompute inside df_ctx
            eval_mask_ctx = (df_ctx.index >= pd.to_datetime(split_start)) & (df_ctx.index <= pd.to_datetime(split_end))
            if eval_mask_ctx.sum() < min_len:
                continue

            first_eval_date = df_ctx.index[eval_mask_ctx][0]
            context_len_effective = int((df_ctx.index < first_eval_date).sum())

            obj_ctx = {
                "ticker": t,
                "segment_id": 0,
                "dates": [d.strftime("%Y-%m-%d") for d in df_ctx.index],
                "sequence": df_ctx["sequence"].astype(float).tolist(),
                "main_features": df_ctx[["sequence", "market_ret_1d"]].astype(float).values.tolist(),
                "macro_features": df_ctx[list(macro.columns)].astype(float).values.tolist(),
                "eval_start_date": first_eval_date.strftime("%Y-%m-%d"),
                "eval_length": int(eval_mask_ctx.sum()),
                "context_length": context_len_effective,
            }
            f_ctx.write(json.dumps(obj_ctx) + "\n")
            n_ctx += 1

    print(
        f"[{split_name}] wrote plain jsonl: {n_plain} rows -> {out_path_plain}"
    )
    print(
        f"[{split_name}] wrote with_context jsonl: {n_ctx} rows -> {out_path_ctx}"
    )


def save_metadata(panel: pd.DataFrame, macro: pd.DataFrame):
    debug_df = pd.DataFrame(index=panel.index)
    debug_df["market_ret_1d"] = panel["market_ret_1d"]
    for c in macro.columns:
        debug_df[c] = panel[c]
    debug_df.to_csv(os.path.join(OUT_ROOT, "metadata", "aligned_market_macro_debug.csv"))

    pd.Series(list(macro.columns), name="macro_feature").to_csv(
        os.path.join(OUT_ROOT, "metadata", "macro_feature_order.csv"),
        index=False
    )

    with open(os.path.join(OUT_ROOT, "metadata", "build_config.json"), "w") as f:
        json.dump(
            {
                "start": START,
                "end": END,
                "market": MARKET,
                "context_length": CONTEXT_LENGTH,
                "fred_series": FRED_SERIES,
                "splits": SPLITS,
            },
            f,
            indent=2,
        )


def main():
    ensure_dirs()

    tickers, firm_rets, market_ret, macro, panel = build_full_panel()
    save_metadata(panel, macro)
    write_eval_csvs(panel)

    for split_name, split_range in SPLITS.items():
        out_path_plain = os.path.join(OUT_ROOT, "eval_jsonl", f"{split_name}.jsonl")
        out_path_ctx = os.path.join(OUT_ROOT, "eval_jsonl_with_context", f"{split_name}_with_context.jsonl")

        write_eval_jsonl_for_split(
            split_name=split_name,
            split_range=split_range,
            firm_rets=firm_rets,
            market_ret=market_ret,
            macro=macro,
            out_path_plain=out_path_plain,
            out_path_ctx=out_path_ctx,
            min_len=32,
        )

    print("Done.")
    print(f"Plain eval JSONLs: {os.path.join(OUT_ROOT, 'eval_jsonl')}")
    print(f"With-context JSONLs: {os.path.join(OUT_ROOT, 'eval_jsonl_with_context')}")
    print(f"Plain eval CSVs: {os.path.join(OUT_ROOT, 'eval_csv')}")
    print(f"With-context CSVs: {os.path.join(OUT_ROOT, 'eval_csv_with_context')}")


if __name__ == "__main__":
    main()